In [1]:
import glob
import os
import pandas as pd
import ast
import traceback
import math

In [2]:
# 固定値定義

# 辞書定義
# code1のProfileType.pyからもってきています
profileIdDic = {
    0x1000:{'name':'レーンリンク情報',      'class':'PROFILETYPE_MPU_ZGM_LANE_LINK_INFO'},
    0x1001:{'name':'区画線情報',        'class':'PROFILETYPE_MPU_ZGM_LANE_DIVISION_LINE'},
    0x1002:{'name':'線形状情報',        'class':'PROFILETYPE_MPU_LINE_GEOMETRY'},
    0x1003:{'name':'破線ペイント情報',     'class':'DotLinePaintInfoProfile'},
    0x1004:{'name':'信号機情報',        'class':'PROFILETYPE_MPU_ZGM_TRAFFIC_LIGHT'},
    0x1005:{'name':'道路標示',          'class':'PROFILETYPE_MPU_ZGM_TRAFFIC_PAINT'},
    0x1006:{'name':'標識情報',          'class':'PROFILETYPE_MPU_ZGM_SIGN_INFO'},
    0x1007:{'name':'停止線',            'class':'PROFILETYPE_MPU_ZGM_STOP_LINE'},
    0x1008:{'name':'曲率情報',          'class':'PROFILETYPE_MPU_ZGM_CURVATURE'},
    0x1009:{'name':'勾配情報',          'class':'PROFILETYPE_MPU_ZGM_SLOPE'},
    0x100A:{'name':'路肩幅員',          'class':'PROFILETYPE_MPU_ZGM_SHOULDER_WIDTH'},
    0x100B:{'name':'LanesGeometry',    'class':'LanesGeometryProfile'},
    0x100C:{'name':'TRANSFER_STS',     'class':'PROFILE_MPU_MAP_DATA_TRANSFER_STS'},
    0x100D:{'name':'BASE_POINT',       'class':'PROFILE_MPU_MAP_DATA_BASE_POINT'},
    0x100E:{'name':'不足データチェック用リスト情報', 'class':'PROFILE_MPU_MAP_ID_LIST'},
    0x1011:{'name':'IVI Stub Info',    'class':'IVIStubInfoProfile'},
    
    0x2000:{'name':'自車位置情報',       'class':'AbsoluteVehiclePositionProfile'},
    
    0x3000:{'name':'レーンリンク情報(US)',  'class':'PROFILETYPE_MPU_US_LANE_LINK_INFO'},
    0x3001:{'name':'Lane Line情報(US)', 'class':'PROFILETYPE_MPU_US_LANE_LINE'},
    0x3002:{'name':'Lane Line形状情報(US)','class':'PROFILETYPE_MPU_US_LANE_LINE_GEOMETRY'},
    0x3003:{'name':'Road Edge情報(US)',  'class':'PROFILETYPE_MPU_US_ROAD_EDGE'},
    0x3004:{'name':'Road Edge形状情報(US)','class':'PROFILETYPE_MPU_US_ROAD_EDGE_GEOMETRY'},
    0x3005:{'name':'信号機情報(US)',     'class':'PROFILETYPE_MPU_US_REGULATORY_TRRAFIC_DEVICE'},
    0x3006:{'name':'道路標示(US)',      'class':'PROFILETYPE_MPU_US_PAVEMENT_MARKING'},
    0x3007:{'name':'標識情報(US)',      'class':'PROFILETYPE_MPU_US_SIGN'},
    0x3008:{'name':'曲率情報(US)',      'class':'PROFILETYPE_MPU_US_CURVATURE'},
    0x3009:{'name':'勾配情報(US)',      'class':'PROFILETYPE_MPU_US_SLOPE'},
    0x300A:{'name':'レーン幅員(US)',      'class':'PROFILETYPE_MPU_US_LANE_WIDTH'},
    0x300B:{'name':'LanesGeometry(US)','class':'LanesGeometryProfile_US'},
    0x300C:{'name':'TRANSFER_STS(US)', 'class':'PROFILE_MPU_MAP_DATA_TRANSFER_STS'},
    0x300D:{'name':'BASE_POINT(US)',   'class':'PROFILE_MPU_MAP_DATA_BASE_POINT'},
    0x300E:{'name':'不足データチェック用リスト情報(US)', 'class':'PROFILE_MPU_MAP_ID_LIST'},
    0x300F:{'name':'IVI Stub Info(US)','class':'IVIStubInfoProfile'}}

# PROFILETYPE_MPU_ZGM_LANE_LINK_INFO or PROFILETYPE_MPU_US_LANE_LINK_INFO
SDLinkageString = {0x0000:'有効', 0x8000:'無効'}

# PROFILE_MPU_MAP_ID_LIST
PROFILE_MPU_MAP_ID_LIST_classifyDic = {0x00:'無効値', 0x01:'周辺情報　(MPU経路)', 0x02:'周辺情報  (ADECU経路)', 0x03:'周辺情報（SDMAP/MPU経路）', 0x04:'周辺情報（SDMAP/ADECU経路）'}

#AbsoluteVehiclePositionProfile
AbsoluteVehiclePositionProfile_classifyDic = {0x00:'MPU', 0x01:'AD'} 

# PROFILE_MPU_MAP_DATA_TRANSFER_STS
PROFILE_MPU_MAP_DATA_TRANSFER_STS_classifyDic = {0x00:'無効値', 0x01:'周辺情報　(MPU経路)', 0x02:'周辺情報  (ADECU経路)', 0x03:'経路情報（IVI経路）', 0x04:'経路情報（MPU経路）', 0x05:'経路情報（ADECU経路）', 0x06:'周辺情報（SDMAP:MPU経路)', 0x07:'周辺情報（SDMAP:ADECU経路)'}
PROFILE_MPU_MAP_DATA_TRANSFER_STS_startEndflagDic = {0x00:'無効値', 0x01:'データ送信開始', 0x02:'データ送信完了'}
PROFILE_MPU_MAP_DATA_TRANSFER_STS_initFlagDic = {0x00:'初期化を実施しない', 0x01:'初期化を実施する'}
PROFILE_MPU_MAP_DATA_TRANSFER_STS_outputClassifyDic = {
        0x01:{0x00:'通常出力', 0x01:'全出力', 0x02:'差分出力(再送)'}, #データ送信開始の場合
        0x02:{0x00:'未完了', 0x01:'完了'}} #データ送信完了の場合

# PROFILETYPE_MPU_US_LANE_LINK_INFO
RouteChangeStatusString = { 0:'変換成功',
                        1:'変換不可（対応する地図が無い）',
                        2:'変換不可（変換候補の評価値が低い）',
                        3:'変換不可（隣のLRPまでの距離が離れすぎている）',
                        4:'変換不可（隣のLRPに接続できるレーンが無い）',
                        5:'変換不可（IVI経路情報異常）',
                        6:'変換不可（その他）'}
RoadTypeString = {  0: '(0) Controlled Access Divided',
                1: '(1) Non-Controlled Access Divided',
                2: '(2) Interchange',
                3: '(3) Ramp',
                4: '(4) Controlled Access Non-Divided',
                5: '(5) Non-Controlled Access Non-Divided',
                6: '(6) Local Divided',
                7: '(7) Local Non-Divided'}
LaneTypeString = {  1: '(1) Normal Driving Lane',
                2: '(2) HOV Lane',
                4: '(4) Bidirectional Lane',
                8: '(8) Bus/Taxi Lane',
                16: '(16) Toll Booth Lane',
                32: '(32) Convertible To Shoulder ',
                64: '(64) Turn Only Lane',
                128: '(128) Other'}

# PROFILETYPE_MPU_ZGM_LANE_DIVISION_LINE
LaneLineTypeDic = { 0:    '(0)線無し',
                    1:    '(1)単線-白実線',
                    2:    '(2)単線-白破線(細)',
                    3:    '(3)単線-白破線(太)',
                    4:    '(4)黄実線',
                    11:    '(11)二重線(同種)-白実線',
                    22:    '(22)二重線(同種)-白破線(細)',
                    44:    '(44)二重線(同種)-黄実線',
                    12:    '(12)二重線(別種)-白実線×白破線(細)',
                    21:    '(21)二重線(別種)-白破線(細)×白実線',
                    14:    '(14)二重線(別種)-白実線×黄実線',
                    41:    '(41)二重線(別種)-黄実線×白実線',
                    24:    '(24)二重線(別種)-白破線(細)×黄実線',
                    42:    '(42)二重線(別種)-黄実線×白破線(細)',
                    414:    '(414)三重線-黄実線×白実線×黄実線',
                    424:    '(424)三重線-黄実線×白破線(細)×黄実線',
                    4114:    '(4114)四重線-黄実線×白実線×白実線×黄実線'
}

# PROFILETYPE_MPU_US_LANE_LINE
LaneLineTypeDic = { 0: '(0)Virtual',
                    1: '(1)Single Solid Paint Line',
                    2: '(2)Single Dashed Paint Line',
                    3: '(3)Double Paint Line, Left Solid, Right Solid',
                    4: '(4)Double Paint Line, Left Dashed, Right Solid',
                    5: '(5)Double Paint Line, Left Solid, Right Dashed',
                    6: '(6)Double Paint Line, Left Dashed, Right Dashed',
                    7: '(7)Triple Paint Line All Solid',
                    8: '(8)Other'}

# カラム定義
# 基本的にProfile Message.csvに書かれている列名そのままですが、最後の3列("Profile_info_0","Profile_info_1","Profile_info_2")については、csvの方では書かれていない(空白)ので、ここで定義しています
prf_col=["logIndex","logTime","ΔlogTime[ms]","length","timeStamp[s]","ΔtimeStamp[ms]","seq","ID","msgcnt","Δmsgcnt","LaneID","分割数","分割番号","Number of Array","Instance ID","Is Retransmission","Is Update","Path Id","Offset[cm]","End Offset[cm]","End Offset Final","Confidence[%]","Standard Deviation","Lane Number","Profile Type","Available","Profile Value","Profile_info_0","Profile_info_1","Profile_info_2"]

# Profile Message ファイル名
message_file_name = "Profile Message"


In [ ]:
# 関数定義
def interpolation(s):
    """
    文字列が“=”で始まっていたら、Noneを返す
    """
    # sがNoneではなく、かつ文字列の先頭が"="でない場合にsを返す
    if s is not None and not str(s).startswith("="):
        return s
    else:
        return None

def str_to_dict(x):
    """
    nanをNoneに変換しつつ、文字列を辞書に変換する。
    """
    # nan かどうかのチェック ---
    # x が float 型で、かつ nan の場合に True となる (注意: xが文字列の "nan" の場合は判定できない)
    if isinstance(x, float) and math.isnan(x):
        return None  # nan の場合は None を返す

    # すでに辞書やリストの場合は、そのまま返す ---
    if isinstance(x, (dict, list)):
        return x

    # x が文字列でない場合は、str()で変換する
    if not isinstance(x, str):
        x = str(x)
        # この時点で x が 'nan' という文字列になった場合
        if x == 'nan':
            return None

    # 空文字列の場合はエラーになるため、適切に処理
    if not x.strip():
        return None

    # 文字列にコロンが含まれていない場合、辞書形式ではないとみなし、元の値を返す
    if ':' not in x:
        return x

    # literal_eval を実行 ---
    try:
        return ast.literal_eval(x)
    except Exception as e:
        print(f"ast.literal_eval でエラーが発生しました: {e}")
        print(f"対象データ: {repr(x)}")
        print(traceback.format_exc())
        return x # エラー時は元の値を返す

def partial_quote_strings_from_dict_loop(df, col_name, definition_dict):
    """
    データフレームの指定された列の文字列に対し、定義辞書の値に部分一致する箇所を
    シングルクォーテーションで囲みます。（ループ版）

    Args:
        df (pd.DataFrame): 対象のデータフレーム
        col_name (str): 対象の列名
        definition_dict (dict): 置換文字列の元となる辞書

    Returns:
        pd.DataFrame: 処理後のデータフレーム
    """
    # 辞書の値（置換したい文字列）を文字長の降順（長いものから）にソートする
    # これにより「変換不可（その他）」が「変換不可」より先に処理される
    sorted_values = sorted(definition_dict.values(), key=len, reverse=True)

    temp_col = df[col_name].astype(str) # 文字列としてコピー
    for value in sorted_values:

        # regex=False で文字列リテラルとして置換
        temp_col = temp_col.str.replace(value, f"'{value}'", regex=False)
    df[col_name] = temp_col
    
    return df

def Profile_info_to_dict(df, col, Profile_Type):

    if Profile_Type == "0x100E": # PROFILE_MPU_MAP_ID_LIST
        df = partial_quote_strings_from_dict_loop(df, col, PROFILE_MPU_MAP_ID_LIST_classifyDic)
        
    elif Profile_Type == "0x2000": # AbsoluteVehiclePositionProfile
        df = partial_quote_strings_from_dict_loop(df, col, AbsoluteVehiclePositionProfile_classifyDic)

    elif Profile_Type == "0x300C": # PROFILE_MPU_MAP_DATA_TRANSFER_STS
        df = partial_quote_strings_from_dict_loop(df, col, PROFILE_MPU_MAP_DATA_TRANSFER_STS_classifyDic)
        df = partial_quote_strings_from_dict_loop(df, col, PROFILE_MPU_MAP_DATA_TRANSFER_STS_startEndflagDic)
        df = partial_quote_strings_from_dict_loop(df, col, PROFILE_MPU_MAP_DATA_TRANSFER_STS_initFlagDic)
        df = partial_quote_strings_from_dict_loop(df, col, PROFILE_MPU_MAP_DATA_TRANSFER_STS_outputClassifyDic)

    elif Profile_Type == "0x1000" or Profile_Type == "0x3000": #  PROFILETYPE_MPU_ZGM_LANE_LINK_INFO or PROFILETYPE_MPU_US_LANE_LINK_INFO
        df = partial_quote_strings_from_dict_loop(df, col, SDLinkageString)

        if Profile_Type == "0x3000": # PROFILETYPE_MPU_US_LANE_LINK_INFO
            df = partial_quote_strings_from_dict_loop(df, col, RouteChangeStatusString)
            df = partial_quote_strings_from_dict_loop(df, col, RoadTypeString)
            df = partial_quote_strings_from_dict_loop(df, col, LaneTypeString)

    df[col]=df[col].str.replace("*","")
    df[col]=df[col].str.replace("   ","", regex=False)
    df[col]=df[col].str.replace("\n$","", regex=True)
    df[col]=df[col].str.replace("\n",",'", regex=True)
    df[col]=df[col].str.replace(":","':", regex=False)

    df[col]=df[col].str.replace("\[error\]","'\[error\]'") #辞書になかったときに格納される
    df[col]=df[col].str.replace("Unknown","'Unknown'") #辞書になかったときに格納される

    df[col]=df[col].str.replace("軽度","経度") #typo修正
    df[col]=df[col].str.replace("座業","座標") #typo修正

    df[col]="{'"+df[col]+"}" 
    df[col]=df[col].apply(str_to_dict)
    return df

def extract_dict_columns(df, dict_col_name):
    """
    DataFrameのカラムに格納されている辞書からkeyとvalueを取得し、
    新しいカラムとしてDataFrameに追加する関数

    Args:
        df (pd.DataFrame): 処理対象のDataFrame
        dict_col_name (str): 辞書が格納されているカラムの名前

    Returns:
        pd.DataFrame: 辞書の内容が展開された新しいDataFrame
    """
    if dict_col_name not in df.columns:
        print(f"Error: Column '{dict_col_name}' not found in the DataFrame.")
        return df

    # 辞書カラムをJSONのように正規化して新しいDataFrameを作成
    normalized_df = pd.json_normalize(df[dict_col_name], errors='ignore')

    # 元のDataFrameと結合
    result_df = pd.concat([df, normalized_df], axis=1)

    return result_df

# ↓この関数でいろいろとやっています
def Profile_info_to_df(df, Profile_Type):

    df_columns = df.columns.tolist()
    profile_columns = ["Profile_info_0", "Profile_info_1", "Profile_info_2"]
    non_profile_columns = list(set(df.columns.tolist()) - set(profile_columns))
    
    # 前処理
    df = df.applymap(interpolation)
    df[non_profile_columns]=df[non_profile_columns].fillna(method='ffill')
    df = df.dropna(subset=profile_columns, how='all')

    # 各Profile Type情報 の抽出
    df = df[df["Profile Type"]==Profile_Type]
    df = df.reset_index(drop=True)

    # 文字列→辞書に変換 ここでTypoの修正もしている
    df = Profile_info_to_dict(df, "Profile_info_0", Profile_Type)
    df = Profile_info_to_dict(df, "Profile_info_1", Profile_Type)
    df = Profile_info_to_dict(df, "Profile_info_2", Profile_Type)

    # 辞書をカラムにする
    df = extract_dict_columns(df, "Profile_info_0")
    Profile_info_0_cols = list(set(df.columns.tolist()) - set(df_columns))
    df[Profile_info_0_cols] = df[Profile_info_0_cols].fillna(method='ffill')

    df = extract_dict_columns(df, "Profile_info_1")
    Profile_info_1_cols = list(set(df.columns.tolist()) - set(df_columns) - set(Profile_info_0_cols))

    df = extract_dict_columns(df, "Profile_info_2")
    Profile_info_2_cols = list(set(df.columns.tolist()) - set(df_columns) - set(Profile_info_0_cols) - set(Profile_info_1_cols))

    subset_colmns = Profile_info_0_cols

    # きれいなテーブルになるように調整
    if len(Profile_info_1_cols)>0:
        subset_colmns = Profile_info_1_cols

    if len(Profile_info_2_cols)>0:
        subset_colmns = Profile_info_2_cols
        df[Profile_info_1_cols] = df[Profile_info_1_cols].fillna(method='ffill')

    # 必要ない行を削除 → これできれいなテーブルになる
    df = df.dropna(subset=subset_colmns, how='all')

    # 不要な列(テーブル内で値が全て同じになる列、カラムとして抽出済みの列)を除外
    output_colmns = [x for x in df.columns.tolist() if x not in ["Profile Type", "Profile Value", "Profile_info_0", "Profile_info_1", "Profile_info_2"]]
    df = df[output_colmns]

    return df


In [14]:
### 入力ディレクトリの指定 ###
csv_save_path = r"/mnt/e/xtech/AD2/JPN/output"
# csv_save_path = r"/mnt/e/xtech/AD2/US/output"
# csv_save_path = "" #エクステック様の環境でのパスを入力

# csvファイルリスト取得
prf_csv_list=sorted(glob.glob(os.path.join(csv_save_path+"/**/"+message_file_name+".csv"), recursive=True))

In [15]:
prf_csv_list #確認用

['/mnt/e/xtech/AD2/JPN/output/J42U_JPN_AD1Next_V001_01ALL_20250131_083508_185/Profile Message.csv',
 '/mnt/e/xtech/AD2/JPN/output/J42U_JPN_AD1Next_V001_01ALL_20250131_083608_186/Profile Message.csv']

In [ ]:
# mainの処理
for prf_csv in prf_csv_list:
    # Profile Message.csvの読み込み
    df_prf = pd.read_csv(prf_csv, header=None, skiprows=[0,1], names=prf_col,usecols=["logTime",'Instance ID', 'Is Retransmission','Path Id', 'Offset[cm]', 'End Offset[cm]', 'Lane Number', 'Profile Type', 'Profile Value',"Profile_info_0","Profile_info_1","Profile_info_2"], encoding="shift_jis",dtype="object", engine='python')

    Profile_Types = df_prf["Profile Type"].apply(interpolation).dropna().unique().tolist()

    for Profile_Type in Profile_Types:
        
        df = df_prf.copy()
        table_name = profileIdDic[int(Profile_Type, 16)]['class']

        print(table_name) # 確認用

        df_out = Profile_info_to_df(df_prf, Profile_Type)
        df_out = df_out.convert_dtypes() #各列の型を適切な型に変換

        directory, filename = os.path.split(prf_csv)
        output_directory = os.sep.join([directory, message_file_name]) 
        os.makedirs(output_directory, exist_ok=True)
        df_out.to_csv(os.sep.join([output_directory, table_name+".csv"]), encoding="cp932")

PROFILE_MPU_MAP_DATA_BASE_POINT
PROFILETYPE_MPU_ZGM_LANE_LINK_INFO
LanesGeometryProfile
PROFILETYPE_MPU_ZGM_LANE_DIVISION_LINE
PROFILETYPE_MPU_LINE_GEOMETRY
DotLinePaintInfoProfile
PROFILETYPE_MPU_ZGM_CURVATURE
PROFILETYPE_MPU_ZGM_SLOPE
PROFILETYPE_MPU_ZGM_SHOULDER_WIDTH
PROFILETYPE_MPU_ZGM_SIGN_INFO
PROFILETYPE_MPU_ZGM_TRAFFIC_PAINT
PROFILETYPE_MPU_ZGM_TRAFFIC_LIGHT
PROFILE_MPU_MAP_DATA_BASE_POINT
PROFILETYPE_MPU_ZGM_LANE_LINK_INFO
LanesGeometryProfile
PROFILETYPE_MPU_ZGM_LANE_DIVISION_LINE
PROFILETYPE_MPU_LINE_GEOMETRY
PROFILETYPE_MPU_ZGM_CURVATURE
PROFILETYPE_MPU_ZGM_SLOPE
PROFILETYPE_MPU_ZGM_SHOULDER_WIDTH
PROFILETYPE_MPU_ZGM_SIGN_INFO
DotLinePaintInfoProfile
